In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [2]:
data = pd.read_csv("..\data\CyberTrollIEEE.csv")

In [3]:
data.head

<bound method NDFrame.head of                                                  content  annotation
0                                 Get fucking real dude.           1
1       She is as dirty as they come  and that crook ...           1
2       why did you fuck it up. I could do it all day...           1
3       Dude they dont finish enclosing the fucking s...           1
4       WTF are you talking about Men? No men thats n...           1
...                                                  ...         ...
22136  @missmayim @Jeopardy A travesty that they chos...           1
22137  @waggykookie They're 11yo cursing, slut shamin...           1
22138  @Cynosure_Nikaaa Just need attention in the na...           1
22139  Y’all hate slut-shaming til you can do it oh okay           1
22140           @Evan_M_G Feels like slut shaming to me.           1

[22141 rows x 2 columns]>

In [4]:
# Initialize the Lemmatizer
lemmatizer = WordNetLemmatizer()

# Preprocessing function to clean the text
def preprocess_text(text):
    text = text.lower()  # Convert to lowercase
    text = ''.join([char for char in text if char not in string.punctuation])  # Remove punctuation
    tokens = word_tokenize(text)  # Tokenize the text
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stopwords.words('english')]  # Lemmatize and remove stopwords
    return ' '.join(tokens)

In [5]:
# Apply text preprocessing to the 'content' column
data['cleaned_content'] = data['content'].apply(preprocess_text)

# Vectorize the text using TF-IDF (can also try CountVectorizer)
vectorizer = TfidfVectorizer()  # Alternatively, use CountVectorizer()
X = vectorizer.fit_transform(data['cleaned_content'])

# Define the labels (1: bullying, 0: not bullying)
y = data['annotation']

In [6]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize KNN with different k values and experiment
k_values = [2,3, 5, 7, 9]  # List of k values to try
best_k = None
best_accuracy = 0
best_model = None

In [7]:
# Try KNN with different k values
for k in k_values:
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train, y_train)
    
    # Predict on the test set
    y_pred = knn_model.predict(X_test)
    
    # Evaluate the model using accuracy, precision, recall, and F1 score
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"KNN with k={k}:")
    print(f"Accuracy: {accuracy:.2f}")
    print(f"Precision: {precision:.2f}")
    print(f"Recall: {recall:.2f}")
    print(f"F1-Score: {f1:.2f}")
    print("-" * 50)
    
    # Track the best model based on accuracy
    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_k = k
        best_model = knn_model

# Output the best KNN model
print(f"Best KNN model is with k={best_k} having accuracy={best_accuracy:.2f}")


KNN with k=2:
Accuracy: 0.78
Precision: 0.94
Recall: 0.54
F1-Score: 0.68
--------------------------------------------------
KNN with k=3:
Accuracy: 0.78
Precision: 0.91
Recall: 0.57
F1-Score: 0.70
--------------------------------------------------
KNN with k=5:
Accuracy: 0.60
Precision: 0.76
Recall: 0.15
F1-Score: 0.25
--------------------------------------------------
KNN with k=7:
Accuracy: 0.60
Precision: 0.79
Recall: 0.14
F1-Score: 0.23
--------------------------------------------------
KNN with k=9:
Accuracy: 0.58
Precision: 0.75
Recall: 0.10
F1-Score: 0.18
--------------------------------------------------
Best KNN model is with k=3 having accuracy=0.78


In [8]:
# Manually test the best KNN model
def predict_bullying(text):
    # Preprocess the input text
    cleaned_text = preprocess_text(text)
    
    # Vectorize the input text using the same vectorizer as during training
    vectorized_text = vectorizer.transform([cleaned_text])
    
    # Use the best trained KNN model to make a prediction
    prediction = best_model.predict(vectorized_text)
    
    # Return the result (1 for bullying, 0 for not bullying)
    if prediction == 1:
        return "Bullying"
    else:
        return "Not Bullying"

In [9]:
# Manually test the model with sample sentences
test_texts = [
    "You are such a loser!",  # Likely bullying
    "Hey, how are you today?",  # Not bullying
    "what a gay guy",
    "your mom's pussy",
      "you are such a fucking cunt",  # Likely bullying
    "That's a great idea!",  # Not bullying
]

for text in test_texts:
    result = predict_bullying(text)
    print(f"Text: {text}\nPrediction: {result}\n")

Text: You are such a loser!
Prediction: Bullying

Text: Hey, how are you today?
Prediction: Not Bullying

Text: what a gay guy
Prediction: Bullying

Text: your mom's pussy
Prediction: Not Bullying

Text: you are such a fucking cunt
Prediction: Bullying

Text: That's a great idea!
Prediction: Not Bullying



In [10]:
from sklearn.svm import SVC

# Initialize the SVM model
svm_model = SVC(kernel='linear')  # You can try other kernels like 'rbf' or 'poly'

# Train the SVM model with the training data
svm_model.fit(X_train, y_train)

# Predict on the test set
y_pred_svm = svm_model.predict(X_test)

# Evaluate the SVM model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)

# Print evaluation metrics for SVM
print("SVM Model Evaluation:")
print(f"Accuracy: {accuracy_svm:.2f}")
print(f"Precision: {precision_svm:.2f}")
print(f"Recall: {recall_svm:.2f}")
print(f"F1-Score: {f1_svm:.2f}")
print("-" * 50)




SVM Model Evaluation:
Accuracy: 0.82
Precision: 0.80
Recall: 0.80
F1-Score: 0.80
--------------------------------------------------


In [11]:
# Manually test the SVM model with sample sentences
def predict_bullying_svm(text):
    # Preprocess the input text
    cleaned_text = preprocess_text(text)
    
    # Vectorize the input text using the same vectorizer as during training
    vectorized_text = vectorizer.transform([cleaned_text])
    
    # Use the trained SVM model to make a prediction
    prediction = svm_model.predict(vectorized_text)
    
    # Return the result (1 for bullying, 0 for not bullying)
    if prediction == 1:
        return "Bullying"
    else:
        return "Not Bullying"

# Manually test the SVM model with sample sentences
test_texts = [
    "You are such a loser!",  # Likely bullying
    "Hey, how are you today?",  # Not bullying
    "I hope you die you mother fuker",  # Likely bullying
    "That's a great idea!",  # Not bullying
]

for text in test_texts:
    result = predict_bullying_svm(text)
    print(f"Text: {text}\nPrediction: {result}\n")

Text: You are such a loser!
Prediction: Bullying

Text: Hey, how are you today?
Prediction: Not Bullying

Text: I hope you die you mother fuker
Prediction: Not Bullying

Text: That's a great idea!
Prediction: Not Bullying



In [15]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)  # 100 trees, you can tune this

# Train the Random Forest model
rf_model.fit(X_train, y_train)

# Predict on the test set
y_pred_rf = rf_model.predict(X_test)

# Evaluate the Random Forest model
accuracy_rf = accuracy_score(y_test, y_pred_rf)
precision_rf = precision_score(y_test, y_pred_rf)
recall_rf = recall_score(y_test, y_pred_rf)
f1_rf = f1_score(y_test, y_pred_rf)

# Print evaluation metrics for Random Forest
print("Random Forest Model Evaluation:")
print(f"Accuracy: {accuracy_rf:.2f}")
print(f"Precision: {precision_rf:.2f}")
print(f"Recall: {recall_rf:.2f}")
print(f"F1-Score: {f1_rf:.2f}")
print("-" * 50)

# Manually test the Random Forest model with sample sentences



Random Forest Model Evaluation:
Accuracy: 0.92
Precision: 0.89
Recall: 0.92
F1-Score: 0.91
--------------------------------------------------


In [21]:
def predict_bullying_rf(text):
    # Preprocess the input text
    cleaned_text = preprocess_text(text)
    
    # Vectorize the input text using the same vectorizer as during training
    vectorized_text = vectorizer.transform([cleaned_text])
    
    # Use the trained Random Forest model to make a prediction
    prediction = rf_model.predict(vectorized_text)
    
    # Return the result (1 for bullying, 0 for not bullying)
    if prediction == 1:
        return "Bullying"
    else:
        return "Not Bullying"

# Manually test the Random Forest model with sample sentences
test_texts = [
    "You are such a loser!",  # Likely bullying
    "Hey, how are you today?",  # Not bullying
    "I hope you die",
    "how mad would you be at me if I were to rape you?",  # Likely bullying
    "That's a great idea!",  # Not bullying
]

for text in test_texts:
    result = predict_bullying_rf(text)
    print(f"Text: {text}\nPrediction: {result}\n")

Text: You are such a loser!
Prediction: Bullying

Text: Hey, how are you today?
Prediction: Not Bullying

Text: I hope you die
Prediction: Not Bullying

Text: how mad would you be at me if I were to rape you?
Prediction: Not Bullying

Text: That's a great idea!
Prediction: Not Bullying



In [22]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, Flatten
from tensorflow.keras.optimizers import Adam

# Define a simple neural network model
model = Sequential([
    Embedding(input_dim=vectorizer.vocabulary_._size, output_dim=128, input_length=X_train.shape[1]),  # Word embeddings
    Dropout(0.2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')  # Binary classification (bullying or not)
])

# Compile the model
model.compile(optimizer=Adam(), loss='binary_crossentropy', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=5, batch_size=32, validation_data=(X_test, y_test), verbose=1)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"TensorFlow Model - Loss: {loss:.2f}, Accuracy: {accuracy:.2f}")

# Manually test the model with sample sentences
def predict_bullying_tf(text):
    # Preprocess and vectorize the input text
    cleaned_text = preprocess_text(text)
    vectorized_text = vectorizer.transform([cleaned_text])
    
    # Use the trained model to make a prediction
    prediction = model.predict(vectorized_text)
    
    if prediction >= 0.5:
        return "Bullying"
    else:
        return "Not Bullying"

# Test with sample sentences
test_texts = [
    "You are such a loser!",
    "Hey, how are you today?",
    "I hope you die",
    "That's a great idea!"
]

for text in test_texts:
    result = predict_bullying_tf(text)
    print(f"Text: {text}\nPrediction: {result}\n")


AttributeError: 'dict' object has no attribute '_size'